# FLUKE Dialogue Contradiction Detection with DeepSeek R1

This notebook evaluates dialogue contradiction detection robustness using DeepSeek R1 open-source reasoning model via OpenRouter API with FLUKE linguistic modifications.

In [1]:
# Standard imports
from datasets import load_dataset
import dspy
import os
import pandas as pd
import json
import glob
import time
import random
from dotenv import load_dotenv
from dspy.evaluate import Evaluate

# Import unified FLUKE utilities
from fluke_reasoning_utils import (
    REASONING_MODELS, REASONING_CONFIGS,
    remove_space, extract_classification_prediction,
    aggregate_results, highlight_drops_and_significance,
    compare_models, append_person
)

/Users/hungthinh/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables
load_dotenv()

# For OpenRouter, we need the OpenRouter API key
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if not openrouter_api_key:
    print("Warning: OPENROUTER_API_KEY not found in environment variables")
    print("Please set your OpenRouter API key in the .env file")

## DeepSeek R1 Configuration

In [3]:
# Select DeepSeek configuration
CONFIG_NAME = 'deepseek'  # Options: 'deepseek', 'deepseek-lite'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Description: {config['description']}")

# Configure DSPy with DeepSeek R1 via OpenRouter
lm = dspy.LM(
    model=MODEL_ID,
    api_key=openrouter_api_key,
    api_base="https://openrouter.ai/api/v1",
    max_tokens=20_000,
    temperature=1  # DeepSeek R1 supports temperature control
)
dspy.configure(lm=lm)

Configuration: deepseek
Model: deepseek-r1 (openrouter/deepseek/deepseek-r1)
Description: Open-source reasoning with DeepSeek R1


## Load Dialogue Data

In [7]:
# Load dialogue dataset
ds = pd.read_json('../../../data/train_dev_test_data/dialog/test.json')
ds = ds.to_dict('records')

print(f"Loaded {len(ds)} dialogue samples")

# Split by contradiction type
samples_with_contradiction = []
samples_no_contradiction = []
for i, x in enumerate(ds):
    if x["is_contradiction"]:
        samples_with_contradiction.append((i, x))
    else:
        samples_no_contradiction.append((i, x))
        
print(f"Contradictions: {len(samples_with_contradiction)}, No contradictions: {len(samples_no_contradiction)}")

# Combine and shuffle
samples = samples_with_contradiction + samples_no_contradiction
random.shuffle(samples)

label_map = {'is_contradiction': 1, 'no_contradiction': 0}

Loaded 4216 dialogue samples
Contradictions: 2108, No contradictions: 2108


In [8]:
# Function to add agent labels to dialogue
def add_agent_labels(dialogue_list):
    """Add agent 0/1 labels to each turn in the dialogue."""
    labeled_dialogue = []
    for i, turn in enumerate(dialogue_list):
        agent_label = f"agent {i % 2}: {turn}"
        labeled_dialogue.append(agent_label)
    return '\n'.join(labeled_dialogue)

# Create examples with agent labels
examples = [
    dspy.Example({
        "dialogue": remove_space(add_agent_labels(r["dialogue"])),
        "label": label_map[r['label']]
    }).with_inputs("dialogue")
    for i, r in samples
]

# Test example
example = examples[59]
print(f"\nExample dialogue with agent labels:")
print(f"{example.dialogue[:500]}...")
print(f"\nLabel: {example.label} ({'Contradiction' if example.label == 1 else 'No contradiction'})")


Example dialogue with agent labels:
agent 0: the braves had a chance to take over first place today, and lost the first game of a DH
agent 1: Oh no. That really stinks! I know how that feels. We like the Red Sox here.
agent 0: Yeah, I actually don't watch a lot of baseball, but I try to follow the world series a little.
agent 1: I watch a lot of baseball but i play basketball more often. in fact I am currently training with my personal coach
agent 0: Cool, do you play professionally? Or are you trying to make a career out of it?
a...

Label: 1 (Contradiction)


## Define Task with DeepSeek R1

In [6]:
class DeepSeekDialogue(dspy.Signature):
    """Given a dialogue with agent labels (agent 0 and agent 1 alternating), determine if the last utterance contradicts the dialogue context. Think step by step about the conversation flow, consistency of statements, and logical coherence. Answer with 1 if it contradicts, 0 if it does not contradict."""
    dialogue = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class DeepSeekDialogueModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(DeepSeekDialogue)

    def forward(self, dialogue):
        return self.prog(dialogue=dialogue)

# Initialize module
deepseek_dialogue = DeepSeekDialogueModule()

# Evaluation metric
def eval_metric(true, prediction, trace=None):
    pred = prediction.label
    parsed_answer = extract_classification_prediction(pred)
    return parsed_answer == str(true.label)

In [14]:
# Test single example
pred = deepseek_dialogue(dialogue=example.dialogue)
print(f"Dialogue: {example.dialogue[:300]}...")
print(f"True Label: {example.label}")
print(f"Prediction: {pred.label}")
print(f"Correct: {eval_metric(example, pred)}")

Dialogue: agent 0: I enjoyed my day the other day, found out something good! Newphew may come to live with us for ayear.
agent 1: Nice! How old is your nephew?
agent 0: 11, he wants to see us for more then 2 weeks a year....
True Label: 0
Prediction: 0
Correct: True


## Evaluate Original Dataset

In [ ]:
# Test size for DeepSeek R1
TEST_SIZE = 200  # Adjust based on API limits and budget
test_examples = examples

print(f"Evaluating {len(test_examples)} examples with DeepSeek R1...")

evaluate = Evaluate(
    devset=test_examples,
    metric=eval_metric,
    num_threads=2,  # Moderate threading for OpenRouter
    display_progress=True,
    display_table=10,
    return_all_scores=True
)

results = evaluate(deepseek_dialogue)

# Save results
items = []
for sample in results['results']:
    items.append({
        'dialog': sample[0]['dialogue'],
        'label': sample[0]['label'],
        'pred': extract_classification_prediction(sample[1]['label'] if 'label' in sample[1] else '0'),
        'raw_output': sample[1]['label'] if 'label' in sample[1] else '0'
    })

df_result = pd.DataFrame(items)
output_file = f'../results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-dialogue.csv'
df_result.to_csv(output_file, index=False)

print(f"\nDeepSeek R1 Accuracy: {results['score']:.3f}")
print(f"Results saved to: {output_file}")

Evaluating 4216 examples with DeepSeek R1...
Average Metric: 2252.00 / 2390 (94.2%):  57%|█████▋    | 2390/4216 [2:02:42<1:52:43,  3.70s/it]

2025/08/18 15:30:32 ERROR dspy.utils.parallelizer: Error for Example({'dialogue': 'agent 0: I love to listen to Elvis. He was an American singer, musician, and actor born in 1935 and died in 1977.\nagent 1: He does play some good music! I didn\'t know he died!\nagent 0: Some people think he faked his death. He is considered one of the most significant cultural icons of the 20th century.\nagent 1: Yes, he had some very big songs. Do you know what type of music he played? I have no idea what you\'d call it.\nagent 0: He started in jazz, then helped start what we now call rock. He was known as the King of Rock and Roll.\nagent 1: Right, he did play a lot of rock and roll. I heard he did a lot of drugs too1\nagent 0: I heard that as well. Elvis released his first single "Heartbreak Hotel" in 1956 which became a number 1 hit in the USA.\nagent 1: That was long before I was born.\nagent 0: Nah, Elvis became popular around 1996.', 'label': 1}) (input_keys={'dialogue'}): expected string or byt

Average Metric: 2699.00 / 2862 (94.3%):  68%|██████▊   | 2863/4216 [2:26:44<1:50:07,  4.88s/it]

2025/08/18 15:54:38 ERROR dspy.utils.parallelizer: Error for Example({'dialogue': "agent 0: wasnt there a movie based on that?\nagent 1: yea it wasnt the best but the 1985 sci fi novel has held up quite well\nagent 0: Do you read much sci-fi?\nagent 1: I thought Ender was so cute.\nagent 0: Oh, how so?\nagent 1: Asa Butterfield is a nice actor\nagent 0: I don't know them, what else have they been in?\nagent 1: I don't know. Harrison Ford is my favorite actor\nagent 0: I loved that movie 'The Fugitive', he was really good in that\nagent 1: He was Han Solo in Star Wars. The bad boy of the galaxy\nagent 0: Yes, the real break-out character\nagent 1: that's my guy! so i take it you like sci fi?", 'label': 0}) (input_keys={'dialogue'}): expected string or bytes-like object, got 'NoneType'. Set `provide_traceback=True` for traceback.


Average Metric: 3977.00 / 4214 (94.4%): 100%|██████████| 4216/4216 [3:34:43<00:00,  3.06s/it]  

2025/08/18 17:02:29 INFO dspy.evaluate.evaluate: Average Metric: 3977.0 / 4216 (94.3%)


,dialogue,example_label,pred_label,eval_metric,label
0,"agent 0: hey there, hows it going agent 1: good, hello how are you...",1.0,1,✔️ [True],NaN
1,agent 0: I didn't know that! What savory flavors are there? agent ...,0.0,0,✔️ [True],NaN
2,"agent 0: Wow! I didn't know they were from New Hope, PA. I used to...",0.0,0,✔️ [True],NaN
3,"agent 0: hey, i hope your night is going well agent 1: hello, than...",1.0,1,✔️ [True],NaN
4,agent 0: When did people start Kayaking? agent 1: Kayaks were crea...,0.0,0,✔️ [True],NaN
5,agent 0: I went to a minor league ball game last night and I saw a...,0.0,0,✔️ [True],NaN
6,agent 0: lol i do not like meat so i am good with cheese pizza age...,1.0,1,✔️ [True],NaN
7,agent 0: hello there! do you fancy a cup of tea? or perhaps a good...,0.0,0,✔️ [True],NaN
8,"agent 0: I felt most lonely when we first moved overseas, my husba...",1.0,1,✔️ [True],NaN
9,"agent 0: hello, tell me about yourself while i take a break from m...",0.0,1,,NaN


KeyError: 'label'

## Evaluate Modifications

In [4]:
def evaluate_modified_set(data, program, max_samples=50):
    """Evaluate on modified dataset with DeepSeek R1."""
    limited_data = data[:max_samples] if len(data) > max_samples else data
    
    mod_examples = [
        dspy.Example({
            "dialogue": "\n".join(data[1]['dialog_context'] + [data[1]['modified_text']]),
            "original_dialogue": "\n".join(data[1]['dialog_context'] + [data[1]['original_text']]),
            "label": int(data[1].get('modified_label', data[1]['label'])),
            "original_label": int(data[1]['label']),
            "type": data[1]['type'] if 'type' in data[1] else None,
            "id": idx
        }).with_inputs("dialogue")
        for idx, data in enumerate(limited_data)
    ]
    
    evaluate = Evaluate(
        devset=mod_examples,
        metric=eval_metric,
        num_threads=2,  # Moderate threading for OpenRouter
        display_progress=True,
        display_table=1,
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Load original predictions
original_pred_file = f'../results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-dialogue.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file)
    original_pred_ds['dialog'] = original_pred_ds['dialog'].apply(remove_space)
    print(f"Loaded original DeepSeek R1 predictions from {original_pred_file}")
else:
    print("Please run original evaluation first")
    original_pred_ds = None

# Test modifications with DeepSeek R1
json_files = glob.glob('../../../data/modified_data/dialogue/*_100.json')

print(f"\nTesting {len(json_files)} modifications with DeepSeek R1...")

for json_file in json_files:
    print(f"\nProcessing: {json_file.split('/')[-1]}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Add agent labels
    data = append_person(data)
    
    # Evaluate with sample limit
    results_mod = evaluate_modified_set(data, deepseek_dialogue, max_samples=150)
    
    # Process results
    items = []
    for sample in results_mod['results']:
        item = {
            'original_dialog': sample[0]['original_dialogue'],
            'modified_dialog': sample[0]['dialogue'],
            'modified_label': sample[0]['label'],
            'original_label': sample[0]['original_label'],
            'modified_pred': extract_classification_prediction(sample[1]['label']),
            'raw_output': sample[1]['label'],
            'type': sample[0]['type']
        }
        
        # Find original prediction
        if original_pred_ds is not None:
            matches = original_pred_ds[original_pred_ds['dialog'] == item['original_dialog']]
            item['original_pred'] = matches.iloc[0]['pred'] if not matches.empty else None
        else:
            item['original_pred'] = None
        
        items.append(item)
    
    df_mod = pd.DataFrame(items)
    mod_name = json_file.split('/')[-1].replace('.json', '')
    output_file = f'../results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-{mod_name}.csv'
    df_mod.to_csv(output_file, index=False)
    
    print(f"Accuracy: {results_mod['score']:.3f}")
    print(f"Saved to: {output_file}")
    
    time.sleep(3)  # Rate limiting for OpenRouter

Loaded original DeepSeek R1 predictions from ../results/dialogue/deepseek-r1-deepseek-0shot-dialogue.csv

Testing 17 modifications with DeepSeek R1...

Processing: casual_100.json
Average Metric: 97.00 / 100 (97.0%): 100%|██████████| 100/100 [00:00<00:00, 186.44it/s]

2025/08/19 10:22:48 INFO dspy.evaluate.evaluate: Average Metric: 97 / 100 (97.0%)


,dialogue,original_dialogue,example_label,original_label,type,id,pred_label,eval_metric
0,I miss my dad. This was his favorite time of the year yea that can...,I miss my dad. This was his favorite time of the year yea that can...,0,0,casual,0,0,✔️ [True]


Accuracy: 97.000
Saved to: ../results/dialogue/deepseek-r1-deepseek-0shot-casual_100.csv


## Chain-of-Thought with DeepSeek R1

In [ ]:
class CoTDeepSeekDialogue(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(DeepSeekDialogue)

    def forward(self, dialogue):
        return self.prog(dialogue=dialogue)

# Test CoT
cot_deepseek_dialogue = CoTDeepSeekDialogue()
pred_cot = cot_deepseek_dialogue(dialogue=example.dialogue)
print("Chain-of-Thought with DeepSeek R1:")
print(f"Dialogue: {example.dialogue[:300]}...")
print(f"\nReasoning: {pred_cot.reasoning if hasattr(pred_cot, 'reasoning') else 'N/A'}")
print(f"\nPrediction: {pred_cot.label}")

## Aggregate Results

In [ ]:
# Aggregate all modification results
result_files = glob.glob(f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')

if result_files:
    results_df = aggregate_results(
        result_files,
        task_name='dialogue_contradiction',
        model_name=f'{MODEL_NAME}-{CONFIG_NAME}'
    )
    
    if not results_df.empty:
        # Display summary
        print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
        print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])
        
        # Save aggregated results
        output_file = f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-DP.csv'
        results_df.to_csv(output_file, index=False)
        print(f"\nAggregated results saved to: {output_file}")
        
        # Display styled results
        styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
        display(styled_df)
else:
    print("No result files found to aggregate")

## Model Comparison

In [ ]:
# Compare DeepSeek R1 with other models
comparison_files = {
    'DeepSeek-R1': f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-dialogue.csv',
    'GPT-5': 'results/dialogue/gpt-5-standard-0shot-dialogue.csv',
    'GPT-4o': 'results/dialogue/gpt4o-0shot-dialogue.csv',
    'Claude-3.5': 'results/dialogue/claude-3-5-sonnet-0shot-dialogue.csv',
    'o3-2025-04-16': 'results/dialogue/o3-2025-04-16-standard-0shot-dialogue.csv',
    'Mixtral-8x22B': 'results/dialogue/mixtral-8x22b-0shot-dialogue.csv'
}

comparison_df = compare_models(comparison_files, task_name='dialogue_contradiction')

if not comparison_df.empty:
    print("\nModel Comparison (including DeepSeek R1):")
    print(comparison_df)
    
    # Calculate DeepSeek R1 performance relative to others
    if 'DeepSeek-R1' in comparison_df['Model'].values:
        deepseek_acc = comparison_df[comparison_df['Model'] == 'DeepSeek-R1']['Accuracy'].values[0]
        
        # Compare with closed-source models
        closed_models = ['GPT-5', 'GPT-4o', 'Claude-3.5', 'o3-2025-04-16']
        closed_accs = comparison_df[comparison_df['Model'].isin(closed_models)]['Accuracy'].values
        
        if len(closed_accs) > 0:
            avg_closed = closed_accs.mean()
            gap = deepseek_acc - avg_closed
            print(f"\nDeepSeek R1 Performance: {deepseek_acc:.3f}")
            print(f"Average of closed-source models: {avg_closed:.3f}")
            print(f"Performance gap: {gap:+.3f} ({gap*100:+.1f}%)")
            print(f"\nNote: DeepSeek R1 is an open-source model competing with proprietary systems")
    
    # Highlight best performer
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]
    
    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No comparison data available")

## DeepSeek R1 Performance Analysis

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Dialogue Contradiction Detection with DeepSeek R1 Complete!")
print(f"{'='*60}")

if 'results' in locals():
    print(f"\nBase accuracy: {results[0]:.3f}")

if 'results_df' in locals() and not results_df.empty:
    avg_row = results_df[results_df['modification'] == 'average'].iloc[0]
    print(f"Average robustness drop: {avg_row['difference']:.3f}")
    print(f"Modifications tested: {len(results_df) - 1}")

print(f"\nDeepSeek R1 Configuration: {config['description']}")
print(f"\nKey advantages of DeepSeek R1:")
print("• Open-source model with strong dialogue understanding")
print("• Cost-effective alternative to proprietary models")
print("• Good multi-turn conversation coherence")
print("• Supports temperature control for output variability")
print("• Available via OpenRouter API")

print(f"\nFiles saved in: results/dialogue/")